# WHR BMI _SNORE

In [1]:
import pandas as pd

df_snore = pd.read_csv(
    "/Users/yoshizawakazuki/Desktop/snoring_metal/raw_gwas_stats/Campos_prePMID_Snoring-mainAnalysis.gz",
    sep="\t"
)


In [2]:
df_whr = pd.read_csv(
    "/Users/yoshizawakazuki/Desktop/Datasets/MVMR/fat-distn.giant.ukbb.meta-analysis.whr.combined.tbl.gz",
    sep="\t"
)

In [8]:
df_whr.head(10)

,SNP,Allele1,Allele2,Freq1,FreqSE,MinFreq,MaxFreq,Effect,StdErr,P-value,Direction
0,rs79565750:T:A,a,t,0.0006,0.0000,0.0006,0.0006,-0.0372,0.0397,0.34940,-?
1,rs186736998:G:T,t,g,0.0003,0.0000,0.0003,0.0003,0.0196,0.0630,0.75630,+?
2,rs7852039:C:T,t,c,0.2189,0.0117,0.2154,0.2583,-0.0008,0.0023,0.71930,-+
3,rs114009005:C:T,t,c,0.0008,0.0000,0.0008,0.0008,0.0322,0.0379,0.39560,+?
4,rs139293132:C:T,t,c,0.0003,0.0000,0.0003,0.0003,-0.0170,0.0621,0.78480,-?
5,rs4336389:A:G,a,g,0.0812,0.0093,0.0766,0.1000,-0.0057,0.0033,0.08283,-+
6,rs188873253:C:T,t,c,0.0002,0.0000,0.0002,0.0002,-0.0497,0.0669,0.45810,-?
7,rs149583963:C:T,t,c,0.0002,0.0000,0.0002,0.0002,-0.0615,0.0725,0.39640,-?
8,rs183344732:G:T,t,g,0.0006,0.0000,0.0006,0.0006,-0.0379,0.0425,0.37300,-?
9,rs6139361:G:A,a,g,0.2235,0.0173,0.1833,0.2309,0.0009,0.0022,0.68750,++


In [3]:
df_bmi = pd.read_csv(
    "/Users/yoshizawakazuki/Desktop/Datasets/MVMR/GCST90179150_buildGRCh37.tsv",
    sep="\t"
)

In [7]:
df_bmi.head(10)

,variant_id,chromosome,base_pair_location,effect_allele,other_allele,effect_allele_frequency,beta,standard_error,z,p_value
0,rs12238997,1,693731,A,G,0.8807,-0.001050,0.003184,-0.329764,0.741578
1,rs144155419,1,717587,A,G,0.0161,-0.000691,0.008204,-0.084188,0.932907
2,rs148120343,1,730087,T,C,0.9453,0.008446,0.004539,1.860706,0.062786
3,rs141242758,1,734349,T,C,0.8752,-0.000899,0.003123,-0.287957,0.773380
4,rs3115860,1,753405,A,C,0.8608,-0.001500,0.002982,-0.503181,0.614837
5,rs2073813,1,753541,A,G,0.1315,0.000492,0.002872,0.171437,0.863880
6,rs12184325,1,754105,T,C,0.0374,-0.000043,0.005442,-0.007879,0.993713
7,rs3131969,1,754182,A,G,0.1425,0.000966,0.002953,0.327084,0.743604
8,rs3131968,1,754192,A,G,0.1411,0.001479,0.002711,0.545533,0.585387
9,rs3131967,1,754334,T,C,0.1424,0.001009,0.002954,0.341476,0.732746


In [5]:
import numpy as np

In [11]:
# ============================================
# 2. WHRデータの標準化
# ============================================

def extract_rsid(snp_string):
    """
    SNP IDからRS IDを抽出
    例: 'rs7956575:T:A' -> 'rs7956575'
        '0:T:A' -> None (RS IDなし)
    """
    if pd.isna(snp_string):
        return None
    
    # コロンで分割
    if ':' in str(snp_string):
        parts = str(snp_string).split(':')
        # 最初の部分がrsで始まる場合
        if parts[0].startswith('rs'):
            return parts[0]
        else:
            return None
    else:
        # そのまま返す
        return str(snp_string)

def standardize_gwas_data(df, col_mapping):
    """GWASデータを標準フォーマットに変換"""
    df_std = df.copy()
    df_std = df_std.rename(columns=col_mapping)
    
    required_cols = ['SNP', 'chr', 'pos', 'effect_allele', 'other_allele', 
                     'eaf', 'beta', 'se', 'pval']
    
    for col in required_cols:
        if col not in df_std.columns:
            df_std[col] = np.nan
    
    df_std = df_std[required_cols]
    
    # WHRデータの場合、SNP IDからRS IDを抽出
    if 'SNP' in df_std.columns:
        print("\nExtracting RS IDs from SNP column...")
        print(f"Original SNP format example: {df_std['SNP'].iloc[0]}")
        df_std['SNP'] = df_std['SNP'].apply(extract_rsid)
        # RS IDがないものを除去
        df_std = df_std[df_std['SNP'].notna()]
        df_std = df_std[df_std['SNP'].str.startswith('rs', na=False)]
        print(f"After extraction: {df_std['SNP'].iloc[0]}")
        print(f"Retained {len(df_std)} SNPs with RS IDs")
    
    # アリルを大文字に統一
    df_std['effect_allele'] = df_std['effect_allele'].str.upper()
    df_std['other_allele'] = df_std['other_allele'].str.upper()
    
    # 数値型に変換
    numeric_cols = ['chr', 'pos', 'eaf', 'beta', 'se', 'pval']
    for col in numeric_cols:
        df_std[col] = pd.to_numeric(df_std[col], errors='coerce')
    
    return df_std

# WHRのカラムマッピング
whr_mapping = {
    'SNP': 'SNP',
    'Allele1': 'effect_allele',  # Effect allele
    'Allele2': 'other_allele',   # Other allele
    'Freq1': 'eaf',              # Effect allele frequency
    'Effect': 'beta',            # Effect size
    'StdErr': 'se',              # Standard error
    'P-value': 'pval'            # P-value
}
whr_std = standardize_gwas_data(df_whr, whr_mapping)

# BMIのカラムマッピング（既存）
bmi_mapping = {
    'variant_id': 'SNP',
    'chromosome':'chr',
    'effect_allele': 'effect_allele',
    'other_allele': 'other_allele',
    'effect_allele_frequency': 'eaf',
    'beta': 'beta',
    'standard_error': 'se',
    'p_value': 'pval'
}
bmi_std = standardize_gwas_data(df_bmi, bmi_mapping)

# Snoreのカラムマッピング（既存）
snore_mapping = {
    'SNP': 'SNP',
    'CHR': 'chr',
    'BP': 'pos',
    'A1': 'effect_allele',
    'A2': 'other_allele',
    'FREQ': 'eaf',
    'BETA': 'beta',
    'SE': 'se',
    'P': 'pval'
}
snore_std = standardize_gwas_data(df_snore, snore_mapping)

print("WHR data shape:", whr_std.shape)
print("BMI data shape:", bmi_std.shape)
print("Snore data shape:", snore_std.shape)

# ============================================
# 3. データのクリーニング
# ============================================

def clean_gwas_data(df, pval_threshold=1.0):
    """GWASデータのクリーニング"""
    df_clean = df.copy()
    
    essential_cols = ['SNP', 'effect_allele', 'other_allele', 'beta', 'se', 'pval']
    df_clean = df_clean.dropna(subset=essential_cols)
    df_clean = df_clean[df_clean['pval'] <= pval_threshold]
    df_clean = df_clean[df_clean['se'] > 0]
    df_clean = df_clean.sort_values('pval').drop_duplicates(subset=['SNP'], keep='first')
    
    return df_clean

# P値の分布を確認
print("\n" + "="*50)
print("P-value distribution check")
print("="*50)

print("\nWHR data P-value distribution:")
if not whr_std['pval'].isna().all():
    print(f"  Min P-value: {whr_std['pval'].min():.2e}")
    print(f"  P < 5e-8: {(whr_std['pval'] < 5e-8).sum()} SNPs")
    print(f"  P < 1e-5: {(whr_std['pval'] < 1e-5).sum()} SNPs")
    print(f"  P < 1e-4: {(whr_std['pval'] < 1e-4).sum()} SNPs")

print("\nBMI data P-value distribution:")
if not bmi_std['pval'].isna().all():
    print(f"  Min P-value: {bmi_std['pval'].min():.2e}")
    print(f"  P < 5e-8: {(bmi_std['pval'] < 5e-8).sum()} SNPs")

# クリーニング（P値閾値を調整）
snore_clean = clean_gwas_data(snore_std)
whr_clean = clean_gwas_data(whr_std, pval_threshold=5e-8)  # GWS
bmi_clean = clean_gwas_data(bmi_std, pval_threshold=5e-8)  # GWS

print("\nAfter cleaning:")
print("Snore data shape:", snore_clean.shape)
print("WHR significant SNPs (P < 5e-8):", whr_clean.shape)
print("BMI significant SNPs (P < 5e-8):", bmi_clean.shape)

# ============================================
# 4. ハーモナイゼーション
# ============================================

def harmonize_alleles(exp_df, out_df):
    """曝露とアウトカムのアリルの向きを揃える"""
    merged = pd.merge(
        exp_df, 
        out_df, 
        on='SNP', 
        suffixes=('_exp', '_out'),
        how='inner'
    )
    
    match = (
        (merged['effect_allele_exp'] == merged['effect_allele_out']) &
        (merged['other_allele_exp'] == merged['other_allele_out'])
    )
    
    flip = (
        (merged['effect_allele_exp'] == merged['other_allele_out']) &
        (merged['other_allele_exp'] == merged['effect_allele_out'])
    )
    
    merged.loc[flip, 'beta_out'] = -merged.loc[flip, 'beta_out']
    merged.loc[flip, 'eaf_out'] = 1 - merged.loc[flip, 'eaf_out']
    
    merged = merged[match | flip]
    
    print(f"  Matched SNPs: {match.sum()}")
    print(f"  Flipped SNPs: {flip.sum()}")
    print(f"  Total harmonized: {len(merged)}")
    
    return merged

print("\nHarmonizing WHR with Snore:")
whr_harm = harmonize_alleles(whr_clean, snore_clean)

print("\nHarmonizing BMI with Snore:")
bmi_harm = harmonize_alleles(bmi_clean, snore_clean)

# ============================================
# 5. MVMR用のデータ準備（Union版）
# ============================================

print("\n" + "="*50)
print("Creating MVMR datasets")
print("="*50)

# Strictバージョン（共通SNPのみ）
common_snps_strict = set(whr_harm['SNP']) & set(bmi_harm['SNP'])
print(f"\nCommon SNPs (strict): {len(common_snps_strict)}")

# Unionバージョン（全SNP）
all_snps_union = set(whr_harm['SNP']) | set(bmi_harm['SNP'])
print(f"Total unique SNPs (union): {len(all_snps_union)}")

# ============================================
# 6. Strict版の作成
# ============================================

whr_strict = whr_harm[whr_harm['SNP'].isin(common_snps_strict)].sort_values('SNP').reset_index(drop=True)
bmi_strict = bmi_harm[bmi_harm['SNP'].isin(common_snps_strict)].sort_values('SNP').reset_index(drop=True)

mvmr_data_strict = pd.DataFrame({
    'SNP': whr_strict['SNP'],
    'chr': whr_strict['chr_exp'],
    'pos': whr_strict['pos_exp'],
    'effect_allele': whr_strict['effect_allele_exp'],
    'other_allele': whr_strict['other_allele_exp'],
    
    # 曝露1: WHR
    'beta_whr': whr_strict['beta_exp'],
    'se_whr': whr_strict['se_exp'],
    'pval_whr': whr_strict['pval_exp'],
    'eaf_whr': whr_strict['eaf_exp'],
    
    # 曝露2: BMI
    'beta_bmi': bmi_strict['beta_exp'],
    'se_bmi': bmi_strict['se_exp'],
    'pval_bmi': bmi_strict['pval_exp'],
    'eaf_bmi': bmi_strict['eaf_exp'],
    
    # アウトカム: Snore
    'beta_snore': whr_strict['beta_out'],
    'se_snore': whr_strict['se_out'],
    'pval_snore': whr_strict['pval_out'],
    'eaf_snore': whr_strict['eaf_out']
})

# ============================================
# 7. Union版の作成
# ============================================

# WHRのデータ準備
whr_for_merge = whr_harm[['SNP', 'chr_exp', 'pos_exp', 'effect_allele_exp', 
                           'other_allele_exp', 'beta_exp', 'se_exp', 
                           'pval_exp', 'eaf_exp', 'beta_out', 'se_out', 
                           'pval_out', 'eaf_out']].copy()
whr_for_merge.columns = ['SNP', 'chr', 'pos', 'effect_allele', 'other_allele',
                          'beta_whr', 'se_whr', 'pval_whr', 'eaf_whr',
                          'beta_snore_whr', 'se_snore_whr', 'pval_snore_whr', 'eaf_snore_whr']

# BMIのデータ準備
bmi_for_merge = bmi_harm[['SNP', 'chr_exp', 'pos_exp', 'effect_allele_exp',
                           'other_allele_exp', 'beta_exp', 'se_exp',
                           'pval_exp', 'eaf_exp', 'beta_out', 'se_out',
                           'pval_out', 'eaf_out']].copy()
bmi_for_merge.columns = ['SNP', 'chr', 'pos', 'effect_allele', 'other_allele',
                          'beta_bmi', 'se_bmi', 'pval_bmi', 'eaf_bmi',
                          'beta_snore_bmi', 'se_snore_bmi', 'pval_snore_bmi', 'eaf_snore_bmi']

# Outer join
mvmr_data_union = pd.merge(whr_for_merge, bmi_for_merge, 
                           on=['SNP'],
                           how='outer',
                           suffixes=('_from_whr', '_from_bmi'))

# chr, pos, alleleの情報を統合
mvmr_data_union['chr'] = mvmr_data_union['chr_from_whr'].fillna(mvmr_data_union['chr_from_bmi'])
mvmr_data_union['pos'] = mvmr_data_union['pos_from_whr'].fillna(mvmr_data_union['pos_from_bmi'])
mvmr_data_union['effect_allele'] = mvmr_data_union['effect_allele_from_whr'].fillna(mvmr_data_union['effect_allele_from_bmi'])
mvmr_data_union['other_allele'] = mvmr_data_union['other_allele_from_whr'].fillna(mvmr_data_union['other_allele_from_bmi'])

# Snoreのbeta/seを統合
mvmr_data_union['beta_snore'] = mvmr_data_union['beta_snore_bmi'].fillna(mvmr_data_union['beta_snore_whr'])
mvmr_data_union['se_snore'] = mvmr_data_union['se_snore_bmi'].fillna(mvmr_data_union['se_snore_whr'])
mvmr_data_union['pval_snore'] = mvmr_data_union['pval_snore_bmi'].fillna(mvmr_data_union['pval_snore_whr'])
mvmr_data_union['eaf_snore'] = mvmr_data_union['eaf_snore_bmi'].fillna(mvmr_data_union['eaf_snore_whr'])

# 曝露のbeta/seで欠損値を埋める
mvmr_data_union['beta_whr'] = mvmr_data_union['beta_whr'].fillna(0)
mvmr_data_union['se_whr'] = mvmr_data_union['se_whr'].fillna(1.0)
mvmr_data_union['pval_whr'] = mvmr_data_union['pval_whr'].fillna(1.0)
mvmr_data_union['eaf_whr'] = mvmr_data_union['eaf_whr'].fillna(0.5)

mvmr_data_union['beta_bmi'] = mvmr_data_union['beta_bmi'].fillna(0)
mvmr_data_union['se_bmi'] = mvmr_data_union['se_bmi'].fillna(1.0)
mvmr_data_union['pval_bmi'] = mvmr_data_union['pval_bmi'].fillna(1.0)
mvmr_data_union['eaf_bmi'] = mvmr_data_union['eaf_bmi'].fillna(0.5)

# 最終的なカラムのみ選択
mvmr_data_union = mvmr_data_union[[
    'SNP', 'chr', 'pos', 'effect_allele', 'other_allele',
    'beta_whr', 'se_whr', 'pval_whr', 'eaf_whr',
    'beta_bmi', 'se_bmi', 'pval_bmi', 'eaf_bmi',
    'beta_snore', 'se_snore', 'pval_snore', 'eaf_snore'
]].sort_values('SNP').reset_index(drop=True)

# ============================================
# 8. データの保存
# ============================================

mvmr_data_strict.to_csv("mvmr_whr_bmi_strict.csv", index=False)
mvmr_data_union.to_csv("mvmr_whr_bmi_union.csv", index=False)

print("\n" + "="*50)
print("FILES SAVED")
print("="*50)
print(f"\nStrict version: mvmr_whr_bmi_strict.csv ({len(mvmr_data_strict)} SNPs)")
print(f"Union version: mvmr_whr_bmi_union.csv ({len(mvmr_data_union)} SNPs)")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"\nWHR significant SNPs (P<5e-8): {(mvmr_data_union['pval_whr'] < 5e-8).sum()}")
print(f"BMI significant SNPs (P<5e-8): {(mvmr_data_union['pval_bmi'] < 5e-8).sum()}")
print(f"\nRecommendation: Use UNION version for better statistical power")


Extracting RS IDs from SNP column...
Original SNP format example: rs79565750:T:A
After extraction: rs79565750
Retained 27384694 SNPs with RS IDs

Extracting RS IDs from SNP column...
Original SNP format example: rs12238997
After extraction: rs12238997
Retained 6206408 SNPs with RS IDs

Extracting RS IDs from SNP column...
Original SNP format example: rs545945172
After extraction: rs545945172
Retained 10443808 SNPs with RS IDs
WHR data shape: (27384694, 9)
BMI data shape: (6206408, 9)
Snore data shape: (10443808, 9)

P-value distribution check

WHR data P-value distribution:
  Min P-value: 4.56e-183
  P < 5e-8: 39709 SNPs
  P < 1e-5: 96056 SNPs
  P < 1e-4: 160674 SNPs

BMI data P-value distribution:
  Min P-value: 0.00e+00
  P < 5e-8: 65558 SNPs

After cleaning:
Snore data shape: (10430975, 9)
WHR significant SNPs (P < 5e-8): (39697, 9)
BMI significant SNPs (P < 5e-8): (65558, 9)

Harmonizing WHR with Snore:
  Matched SNPs: 18915
  Flipped SNPs: 20689
  Total harmonized: 39604

Harmoni